In [ ]:
import time
t0 = time.time()

print(time.time() - t0)

import os
print(os.getcwd())

!ls -la ./use_Hybrid_Data


1.621246337890625e-05
/Users/debajyotihazra/Documents/Modify RAG learning/HYbrid RAG
ls: ./use_Hybrid_Data: No such file or directory


In [ ]:
# Loader load 
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader('Why_Language_Models_Hallucinate_Explainer.pdf')
pages = loader.load()

# Chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap = 120)
texts = text_spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

# VectorDB create 
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="./use_hybrid_Data")
collection = client.get_or_create_collection(name="HYbrid_RAG",embedding_function=embedding_function)
embedding_function(["test sentence"])

if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )

import string
from rank_bm25 import BM25Okapi
tokens = [c.split() for c in chunks]
bm250 = BM25Okapi(tokens)


In [ ]:
from dotenv import load_dotenv 
import os 
load_dotenv()
key = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq 
cha_model = ChatGroq(model_name = 'llama-3.1-8b-instant' , temperature=0,api_key=key)

collection.count()


13

In [ ]:
def Hybrid_Function(query:str):
    result = collection.query(query_texts=[query],n_results=5)
    doc_= result['documents'][0]
    distance = result['distances'][0]

    thresold = 1.0 

    dense_filter = []
    for doc , d in zip(doc_, distance):
        if thresold > d :
            dense_filter.append(doc)
    # Find out the score : 
        rm_score = bm250.get_scores(query.split())
    # Find out the closest index 
    def get_top_index(score , k=10):
        index = list(enumerate(score)) # It basically add index on score area like { 1 : 0.23 , 2 : 0.25}
        index_score = sorted(index ,key=lambda x : x[1],reverse=True) # short them based on the value 
        return [i for i , s in index_score[:k]] # pull out nearest chunks one by one 
    top_index = get_top_index(score=rm_score , k=10)
    #---------------------------------------------------------------------------------------------
    sparse_index = [chunks[i] for i in top_index]
    
# _____________________________________________ 
    rrf_rank = {}

    for rank , doc in enumerate(dense_filter):
        rrf_rank[doc]=rrf_rank.get(doc,0)+1/ (60 + rank)
    for rank , doc in enumerate(sparse_index):
        rrf_rank[doc]=rrf_rank.get(doc,0)+1/ (60 + rank)
    merge = sorted(rrf_rank.items(),key=lambda x : x[1],reverse=True)
    top_docs = [doc for doc , _ in merge[:5]]

    if not top_docs:
        return "NOT RELEVENT TOPICS"
    return "\n\n".join(top_docs)


In [ ]:
qustion = "Why Model Halucinate ?"
answers = Hybrid_Function(query=qustion)
prompt = f"""
you are a reliable LLM so , provide me answers based on the content dont use outside content any time , 
content : {answers}
qustion : {qustion}
"""
cha_model.invoke(prompt)


AIMessage(content='Based on the provided content, I will answer your question:\n\n**Why Language Models Hallucinate: A Statistical Account**\n\nAccording to the content, language models hallucinate due to the following reasons:\n\n1. **Poor-model errors**: Tasks such as counting characters or tokens are mismatched to how subword-based models represent text, producing errors that are structural rather than statistical.\n2. **Out-of-distribution (OOD) inputs**: Queries far outside the training distribution leave the model without reliable statistical anchors.\n3. **Computational hardness**: Some tasks (such as decrypting ciphertext without a key) are provably hard regardless of model competence.\n\n**Why Post-Training Does Not Solve It**\n\nPost-training methods such as reinforcement learning from human feedback (RLHF) and direct preference optimization (DPO) do not solve the hallucination problem because:\n\n* The technique is not the failure point; the grading is.\n* Prevailing benchma